# 🛒 حل مشروع التجارة الإلكترونية المتكامل (E-Commerce Solution)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load data
df = pd.read_csv('datasets/ecommerce_transactions.csv')
print("Shape:", df.shape)
print(df.head())

### 🛠️ الجزء الأول: تنظيف البيانات والتواريخ

In [ ]:
# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', format='mixed')
df = df.dropna(subset=['order_date'])

# Remove duplicate rows
df = df.drop_duplicates()
print("After cleaning shape:", df.shape)

### 📊 الجزء الثاني: تحليل RFM

In [ ]:
max_date = df['order_date'].max()
rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (max_date - x.max()).days,
    'order_id': 'nunique',
    'total_amount': 'sum'
}).rename(columns={'order_date': 'Recency', 'order_id': 'Frequency', 'total_amount': 'Monetary'})

print(rfm.head())

### 🤖 الجزء الثالث: التنبؤ برحيل العملاء ومقارنة الموديلات

In [ ]:
# Churn definition: Recency > 90 days
rfm['Churn'] = (rfm['Recency'] > 90).astype(int)

X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results[name] = {"Accuracy": acc, "F1-Score": f1}
    print(f"=== {name} ===")
    print(classification_report(y_test, preds))

# Plot Comparison
res_df = pd.DataFrame(results).T
res_df.plot(kind='bar', figsize=(8, 5))
plt.title('Model Performance Comparison (Churn Prediction)')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('solutions/ecommerce_model_comparison.png')
plt.show()